# Bye-Bye Lanternfly — Model Training

Train the adult spotted lanternfly YOLOv8 detector in Google Colab. This notebook keeps model training separate from the Flask website.

**Expected dataset layout** (edit `DATASET_DIR` below if yours differs):

```text
lanternfly_v4_fixed/
├── data.yaml
├── train/images, train/labels
├── val/images, val/labels
└── test/images, test/labels
```

The final trained checkpoint is copied to `models/best.pt` in Google Drive. Download that file and place it in this repository's `models/` folder for the website.


In [ ]:
# Run this notebook in Google Colab.
!pip -q install ultralytics==8.4.202 pyyaml


In [ ]:
from pathlib import Path
import shutil
import yaml
from ultralytics import YOLO

# EDIT THESE TWO PATHS if your Google Drive folders are named differently.
from google.colab import drive
drive.mount("/content/drive")

DATASET_DIR = Path("/content/lanternfly_v4_fixed")
RUNS_DIR = Path("/content/drive/MyDrive/ByeByeLanternfly/training_runs")
RUN_NAME = "lanternfly_yolov8n"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

DATA_YAML = DATASET_DIR / "data.yaml"
assert DATA_YAML.exists(), f"Could not find {DATA_YAML}. Update DATASET_DIR."
print(DATA_YAML.read_text())


## Train

This uses the nano YOLOv8 architecture as a practical baseline. If you are reproducing an existing trained model, change the run name so you do not overwrite prior results. Adjust epochs only when intentionally running a new experiment.


In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,              # Colab GPU
    workers=2,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    seed=42,
    plots=True,
    verbose=True
)


## Save the best checkpoint

The website does not need the dataset or training run. It only needs `best.pt`.


In [ ]:
BEST_SOURCE = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
MODEL_OUTPUT_DIR = Path("/content/drive/MyDrive/ByeByeLanternfly/models")
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_DESTINATION = MODEL_OUTPUT_DIR / "best.pt"

assert BEST_SOURCE.exists(), f"Training checkpoint not found: {BEST_SOURCE}"
shutil.copy2(BEST_SOURCE, BEST_DESTINATION)

print(f"Saved: {BEST_DESTINATION}")
print(f"Size: {BEST_DESTINATION.stat().st_size / (1024 * 1024):.2f} MB")


In [ ]:
# Optional: download best.pt directly from Colab to your computer.
from google.colab import files
files.download(str(BEST_DESTINATION))
